# Механизмы внимания и трансформеры

**AI3 · NLP Deep Dive** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

Одной из самых важных задач в области обработки естественного языка (NLP) является **машинный перевод**, ключевая задача, лежащая в основе таких инструментов, как Google Translate. В этом разделе мы сосредоточимся на машинном переводе или, более широко, на любой задаче *последовательность-к-последовательности* (также называемой **трансдукцией предложений**).

С помощью RNN задачи последовательность-к-последовательности реализуются двумя рекуррентными сетями, где одна сеть, **кодировщик**, преобразует входную последовательность в скрытое состояние, а другая сеть, **декодировщик**, разворачивает это скрытое состояние в результат перевода. Однако у этого подхода есть несколько проблем:

* Финальное состояние сети-кодировщика с трудом запоминает начало предложения, что ухудшает качество модели для длинных предложений.
* Все слова в последовательности оказывают одинаковое влияние на результат. На практике же отдельные слова во входной последовательности часто имеют большее влияние на выходные последовательности, чем другие.

**Механизмы внимания** предоставляют способ взвешивания контекстного влияния каждого входного вектора на каждое предсказание выхода RNN. Это реализуется путем создания "ярлыков" между промежуточными состояниями входной RNN и выходной RNN. Таким образом, при генерации выходного символа y<sub>t</sub> мы учитываем все скрытые состояния входа h<sub>i</sub>, с различными весовыми коэффициентами α<sub>t,i</sub>.

![Изображение, показывающее модель кодировщик/декодировщик с аддитивным слоем внимания](ms-assets/ai/encoder-decoder-attention.7a726296894fb567.webp)

> Модель кодировщик-декодировщик с механизмом аддитивного внимания из [Bahdanau et al., 2015](https://arxiv.org/pdf/1409.0473.pdf), цитируется из [этого блога](https://lilianweng.github.io/lil-log/2018/06/24/attention-attention.html)

Матрица внимания {α<sub>i,j</sub>} представляет степень, в которой определенные входные слова участвуют в генерации данного слова в выходной последовательности. Ниже приведен пример такой матрицы:

![Изображение, показывающее пример выравнивания, найденного RNNsearch-50, взято из Bahdanau - arxiv.org](ms-assets/ai/bahdanau-fig3.09ba2d37f202a6af.webp)

> Рисунок из [Bahdanau et al., 2015](https://arxiv.org/pdf/1409.0473.pdf) (Рис.3)

Механизмы внимания ответственны за значительную часть текущего или близкого к текущему уровня развития NLP. Однако добавление внимания значительно увеличивает количество параметров модели, что привело к проблемам масштабирования с RNN. Основным ограничением масштабирования RNN является то, что рекуррентная природа моделей затрудняет пакетную обработку и параллелизацию обучения. В RNN каждый элемент последовательности должен обрабатываться в последовательном порядке, что делает параллелизацию сложной.

![Кодировщик-декодировщик с вниманием](ms-assets/ai/lessons/5-NLP/18-Transformers/images/EncDecAttention.gif)

> Рисунок из [Google's Blog](https://research.googleblog.com/2016/09/a-neural-network-for-machine.html)

Применение механизмов внимания в сочетании с этим ограничением привело к созданию современных моделей трансформеров, таких как BERT и GPT-3, которые мы знаем и используем сегодня.



## Модели трансформеров

Одна из ключевых идей трансформеров заключается в том, чтобы избежать последовательной природы RNN и создать модель, которая может быть параллелизована во время обучения. Это достигается за счет реализации двух идей:

* позиционное кодирование
* использование механизма самовнимания для захвата паттернов вместо RNN (или CNN) (именно поэтому статья, представляющая трансформеры, называется *[Attention is all you need](https://arxiv.org/abs/1706.03762)*)

### Позиционное кодирование/встраивание

Идея позиционного кодирования следующая. 
1. При использовании RNN относительное положение токенов представляется количеством шагов, и поэтому не требует явного представления. 
2. Однако при переходе к вниманию необходимо знать относительные позиции токенов в последовательности. 
3. Чтобы получить позиционное кодирование, мы дополняем нашу последовательность токенов последовательностью позиций токенов в последовательности (т.е. последовательностью чисел 0,1, ...).
4. Затем мы смешиваем позицию токена с вектором встраивания токена. Для преобразования позиции (целого числа) в вектор можно использовать разные подходы:

* Обучаемое встраивание, аналогичное встраиванию токенов. Это подход, который мы рассматриваем здесь. Мы применяем слои встраивания как к токенам, так и к их позициям, получая векторы встраивания одинаковых размеров, которые затем складываем.
* Фиксированная функция позиционного кодирования, предложенная в оригинальной статье.

![](ms-assets/ai/pos-embedding.e41ce9b6cf6078af.webp)

> Изображение автора

Результат, который мы получаем с позиционным встраиванием, включает как оригинальный токен, так и его позицию в последовательности.

### Многоголовое самовнимание

Далее нам нужно захватить некоторые паттерны внутри нашей последовательности. Для этого трансформеры используют механизм **самовнимания**, который по сути является вниманием, применяемым к одной и той же последовательности как входу, так и выходу. Применение самовнимания позволяет учитывать **контекст** внутри предложения и видеть, какие слова взаимосвязаны. Например, это позволяет понять, к каким словам относятся местоимения, такие как *it*, и учитывать контекст:

![](ms-assets/ai/CoreferenceResolution.861924d6d384a7d6.webp)

> Изображение из [Google Blog](https://research.googleblog.com/2017/08/transformer-novel-neural-network.html)

В трансформерах используется **многоголовое внимание**, чтобы дать сети возможность захватывать несколько различных типов зависимостей, например, долгосрочные и краткосрочные отношения между словами, кореференции и другие.

[Блокнот TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/18-Transformers/TransformersTF.ipynb) содержит больше деталей о реализации слоев трансформера.

### Внимание кодировщика-декодировщика

В трансформерах внимание используется в двух местах:

* Для захвата паттернов внутри входного текста с помощью самовнимания.
* Для выполнения перевода последовательности — это слой внимания между кодировщиком и декодировщиком.

Внимание кодировщика-декодировщика очень похоже на механизм внимания, используемый в RNN, как описано в начале этого раздела. Этот анимированная диаграмма объясняет роль внимания кодировщика-декодировщика.

![Анимированный GIF, показывающий, как выполняются оценки в моделях трансформеров.](ms-assets/ai/lessons/5-NLP/18-Transformers/images/transformer-animated-explanation.gif)

Поскольку каждая входная позиция сопоставляется независимо с каждой выходной позицией, трансформеры могут лучше параллелизоваться, чем RNN, что позволяет создавать гораздо более крупные и выразительные языковые модели. Каждая голова внимания может быть использована для изучения различных отношений между словами, что улучшает задачи обработки естественного языка.



## BERT

**BERT** (Bidirectional Encoder Representations from Transformers) — это очень большая многослойная сеть трансформеров с 12 слоями для *BERT-base* и 24 для *BERT-large*. Модель сначала предварительно обучается на большом корпусе текстовых данных (WikiPedia + книги) с использованием обучения без учителя (предсказание замаскированных слов в предложении). Во время предварительного обучения модель усваивает значительные уровни понимания языка, которые затем можно использовать с другими наборами данных с помощью тонкой настройки. Этот процесс называется **перенос обучения**.

![картинка с http://jalammar.github.io/illustrated-bert/](ms-assets/ai/jalammarBERT-language-modeling-masked-lm.34f113ea5fec4362.webp)

> Изображение [источник](http://jalammar.github.io/illustrated-bert/)



## Упражнения: Трансформеры

Продолжите обучение в следующих блокнотах:

* [Трансформеры в PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/18-Transformers/TransformersPyTorch.ipynb)
* [Трансформеры в TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/18-Transformers/TransformersTF.ipynb)



## Заключение

В этом уроке вы узнали о трансформерах и механизмах внимания — всех необходимых инструментах в наборе NLP. Существует множество вариаций архитектур трансформеров, включая BERT, DistilBERT, BigBird, GPT и другие, которые можно тонко настроить. Пакет [HuggingFace](https://github.com/huggingface/) предоставляет репозиторий для обучения многих из этих архитектур как с PyTorch, так и с TensorFlow.



## Обзор и самостоятельное изучение

* [Пост в блоге](https://mchromiak.github.io/articles/2017/Sep/12/Transformer-Attention-is-all-you-need/), объясняющий классическую статью [Attention is all you need](https://arxiv.org/abs/1706.03762) о трансформерах.
* [Серия постов в блоге](https://towardsdatascience.com/transformers-explained-visually-part-1-overview-of-functionality-95a6dd460452) о трансформерах, подробно объясняющая архитектуру.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Causal masked attention

●●● · ~30 min

> Scaled dot-product attention computes scores S = QKᵀ/√d, turns each row into weights with softmax, and returns weights·V. **Causal** (decoder) attention additionally forbids position i from seeing j > i: set those scores to −∞ before softmax so their weight becomes exactly 0. That's an upper-triangular mask — `np.triu(..., k=1)`.

Implement `causal_attention(Q, K, V)` for n positions × d dims: mask future scores (j > i) to −∞, softmax each row of QKᵀ/√d, return the (n, d_v) weighted sum of V. Subtract the row max before exp for stability.

<details><summary>Hint</summary>

mask = np.triu(np.ones((n, n), bool), 1); S = np.where(mask, -np.inf, S); then row-softmax and multiply by V.

</details>


In [ ]:
import numpy as np

def causal_attention(Q, K, V):
    Q = np.asarray(Q, dtype=float)
    # your code
    return np.asarray(V, dtype=float)

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(2, 9)); d = int(rng.integers(2, 7)); dv = int(rng.integers(2, 7))\n    Q = rng.normal(0, 1, (n, d)); K = rng.normal(0, 1, (n, d)); V = rng.normal(0, 1, (n, dv))\n    if rng.random() < 0.45:\n        # huge last-row logits: softmax without max-subtraction -> inf/inf = NaN\n        Q[n - 1] = 100.0\n        K = np.ones((n, d)) * 10.0\n    return {\"Q\": Q, \"K\": K, \"V\": V}\n\ndef causal_attention(Q, K, V):\n    Q = np.asarray(Q, dtype=float)\n    K = np.asarray(K, dtype=float)\n    V = np.asarray(V, dtype=float)\n    n = Q.shape[0]\n    S = Q @ K.T / math.sqrt(Q.shape[1])\n    mask = np.triu(np.ones((n, n), dtype=bool), k=1)\n    S = np.where(mask, -np.inf, S)\n    S = S - S.max(axis=1, keepdims=True)\n    E = np.exp(S)\n    W = E / E.sum(axis=1, keepdims=True)\n    return W @ V", "causal_attention", 20)

### Task 2 — Encoder-decoder cross-attention

●●● · ~15 min

> In an encoder–decoder Transformer the decoder attends to the encoder's outputs: the **queries** come from decoder states while **keys and values** come from the encoder. Scores S = QKᵀ/√d have shape (n_dec, n_enc) and there is NO causal mask — every decoder position may attend to every encoder position. Softmax each row, then multiply by V to get (n_dec, d_v).

Implement `cross_attention(Q_dec, K_enc, V_enc)`: `Q_dec` is (n_dec, d), `K_enc` is (n_enc, d), `V_enc` is (n_enc, d_v). Compute S = Q @ K_enc.T / √d, softmax each row (subtract the row max first for stability), and return the (n_dec, d_v) weighted sum W @ V. No masking of any kind.

<details><summary>Hint</summary>

Same pipeline as causal attention minus the triu mask: S = Q @ K.T / sqrt(d); row-softmax; W @ V.

</details>


In [ ]:
import numpy as np

def cross_attention(Q_dec, K_enc, V_enc):
    Q = np.asarray(Q_dec, dtype=float)
    # your code
    return np.asarray(V_enc, dtype=float)

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    nd = int(rng.integers(2, 8)); ne = int(rng.integers(2, 9))\n    d = int(rng.integers(2, 7)); dv = int(rng.integers(2, 7))\n    Q = rng.normal(0, 1, (nd, d)); K = rng.normal(0, 1, (ne, d)); V = rng.normal(0, 1, (ne, dv))\n    if rng.random() < 0.45:\n        # huge logits: softmax without max-subtraction -> inf/inf = NaN\n        Q[0] = 100.0\n        K = np.ones((ne, d)) * 10.0\n    return {\"Q_dec\": Q, \"K_enc\": K, \"V_enc\": V}\n\ndef cross_attention(Q_dec, K_enc, V_enc):\n    Q = np.asarray(Q_dec, dtype=float)\n    K = np.asarray(K_enc, dtype=float)\n    V = np.asarray(V_enc, dtype=float)\n    S = Q @ K.T / math.sqrt(Q.shape[1])\n    S = S - S.max(axis=1, keepdims=True)\n    E = np.exp(S)\n    W = E / E.sum(axis=1, keepdims=True)\n    return W @ V", "cross_attention", 20)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/18-Transformers)
- [Учебник ML (ШАД): трансформеры](https://contest.yandex.ru/tracks/ml/deep-learning-architectures/transformers)